In [1]:
import sys, os
sys.path.append(os.path.abspath('../../'))
import setup_path

In [2]:
import numpy as np
import pandas
from collections import defaultdict
import random
import time

from docplex.mp.model import Model
import cplex

from src.problems.MCLP import get_neighbors, MCLP
from src.utils.cplex_helpers import solution, cont_solution, check_feasibility, WS_parameters
from src.utils.OQUBO import DocplexModeltoQUBO, QUBOtoIsingModel
from src.utils.helpers import compute_expect, compute_cost, BruteForceSolutions

import datetime
import yfinance as yf

import qiskit
from qiskit_aer import AerSimulator
from qiskit.quantum_info import SparsePauliOp
from scipy.optimize import minimize
from qiskit import QuantumCircuit, transpile
from qiskit.circuit import Parameter
from qiskit.quantum_info import Statevector, DensityMatrix, Pauli
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

from qiskit.circuit.library import efficient_su2
from qiskit.circuit import ParameterVector

In [3]:
with open("data.txt", "r") as file:
    instances_dict = eval(file.read())

In [4]:
def min_QAOA(model, ising_model, p, backend, LR=False, WS_params=None, iters=500):
    enrgs = []
    probs_opt = []
    def Cost_function(params, ising_model, model, quantum_circuit, backend, p, gammas, betas, LR=False, minimize=True):
        global n_iters
        
        if LR:
            gamma_vals = np.arange(1, p + 1) * params[0]/p # Annealing liear ramp schecule (cost hamiltonian)
            beta_vals = np.arange(1, p + 1)[::-1] * params[1]/p # Annealing liear ramp schedule (mixer term)
        else:
            gamma_vals = params[:p]
            beta_vals =  params[p:]
        
        bind_dict = {gammas[i]: gamma_vals[i] for i in range(len(gamma_vals))}
        bind_dict.update({betas[i]: beta_vals[i]   for i in range(len(beta_vals))})
        qc_exec = quantum_circuit.assign_parameters(bind_dict)
    
        shots = 1000
        result = backend.run(qc_exec, shots=shots).result().get_counts() 
    
        opt_sol = solution(model)
        prob_opt = result.get(opt_sol,0)/shots
        energy_qubo = compute_cost(result, ising_model)
        enrgs.append(energy_qubo)
        probs_opt.append(prob_opt)
        if minimize:
            return energy_qubo
        else: 
            return energy_qubo, result
    
    n_qubits = model.number_of_variables
    gammas = ParameterVector('γ', length=p)
    betas = ParameterVector('β', length=p)
    
    hamiltonian, ising_constant = ising_model
    
    ansatz = QuantumCircuit(n_qubits)

    if WS_params:
        for i, param in enumerate(WS_params):
            ansatz.ry(param, i)
    else:
        ansatz.h(range(n_qubits))
        
    for ii in range(p):
        for qbits, value in hamiltonian.items():
            if qbits[0] == qbits[1]:
                ansatz.rz(2*gammas[ii]*float(value), qbits[0])
        for qbits, value in hamiltonian.items():
            if qbits[0] != qbits[1]:
                ansatz.rzz(2*gammas[ii]*float(value), *qbits)
        if WS_params:
            for i, param in enumerate(WS_params):
                ansatz.ry(-param, i)
            ansatz.rz(-2*betas[ii], range(n_qubits))
            for i, param in enumerate(WS_params):
                ansatz.ry(param, i)
        else:
            ansatz.rx(-2*betas[ii], range(n_qubits))
            
    ansatz = ansatz.reverse_bits()
    ansatz.measure_all()
    gates = dict(ansatz.count_ops())
    
    # Transpile the ansatz circuit for the given backend
    qc = transpile(ansatz, backend=backend)
    
    if LR:
        delta_gamma = np.random.random() 
        delta_beta = np.random.random() 
        params = [delta_gamma, delta_beta]
        bounds_params = [(0.0000001, 0.9), (0.0000001, 0.9)] 
    else:
        gammas0 = np.random.normal(0, 1, size=p)
        betas0 = np.random.normal(0, 1, size=p)
        params = np.concatenate([gammas0, betas0])
        bounds_params = [(0.0000001, np.pi)] * (2*p)

    res = minimize(Cost_function,
            params,
            args=(ising_model, model, qc, backend, p, gammas, betas, LR),
            bounds=bounds_params,
            method="cobyla",
            options={"maxiter": iters})

    params = res.x
    f_energy, qaoa_result = Cost_function(params, ising_model, model, qc, backend, p, gammas, betas, LR, minimize=False)
    
    return res, enrgs, probs_opt, qaoa_result, f_energy, gates

In [5]:
def run_MCLP(size_grid, instances_dict, p, N, alpha, slack_vars=False, WS=False, LR=False):
    shots = 1000

    data = {}
    data['r'] = []
    data['probs'] = []
    data['success_counter'] = []
    data['qaoa_FR'] = []
    data['t_ws'] = []
    data['gates'] = []
    data['t_qaoa'] = []
    data['t_total'] = []
    data['n_fev'] = []
    data['n_qubits'] = []
    data['size_grid'] = size_grid
    data['gse'] = []
    data['qaoa_e'] = []
    data['opt_cost'] = []
    for n in n_grid:
        r_n = []
        probs_n = []
        qaoa_fes = []
        t_ws = []
        t_qaoa = []
        t = []
        n_fev = []
        gse = []
        opt_cost = []
        qaoa_e = []
        count = 0
        print(f'{n} places')
        for l in range(N):
            print(f'{l} experiment')
            J = instances_dict[str(n)][l][0]
            I = instances_dict[str(n)][l][1]
            S = 1
            P = int(np.ceil(instances_dict[str(n)][l][2]/2))
            model = MCLP(J, I, P, S, version= 'COP', slack_vars=slack_vars)
            n_qubits = model.number_of_variables
            print(f'{n_qubits} qubits')
            
            backend = AerSimulator()

            t_ws_t = 0
            if WS:
                cont_model = MCLP(J, I, P, S, version= 'CONT', slack_vars=slack_vars)
                t_ws_i = time.time()
                cont_sol = cont_solution(cont_model)
                WS_params = WS_parameters(cont_sol)
                t_ws_f = time.time()
                t_ws_t = t_ws_f-t_ws_i
            else:
                WS_params = None
            t_ws.append(t_ws_t)
            
            ising_model = QUBOtoIsingModel(
                model,
                include='Total', 
                normalize=True, 
                obj_normalize=True, 
                pen_normalize=False, 
                UP=not slack_vars,
                alpha=alpha, 
                gamma=0, 
                precision=6
            )
    
            t_qaoa_i = time.time()
            res, energies, probs, qaoa_result, qaoa_energy, gates = min_QAOA(model, ising_model, p, backend, LR=LR, WS_params=WS_params)
            qaoa_e.append(qaoa_energy)
            n_fev.append(res.nfev)
            t_qaoa_f = time.time()
            t_qaoa_t = t_qaoa_f - t_qaoa_i
            t_qaoa.append(t_qaoa_t)
            t.append(t_ws_t + t_qaoa_t)
            opt_sol = solution(model)
            opt_sol_cost = compute_cost({opt_sol:1}, ising_model)
            opt_cost.append(opt_sol_cost)
            probs_n.append(qaoa_result.get(opt_sol,0)/shots)

            e_min = 0
            brute_force_list = list(BruteForceSolutions(model, alpha, 0, 
                                    UP=not slack_vars, 
                                    include='Total', 
                                    normalize=True, 
                                    obj_normalize=True,
                                    pen_normalize=False).items())
            s_min = brute_force_list[0][0]
            e_min = brute_force_list[0][1]
            e_max = brute_force_list[-1][1]
            if s_min==opt_sol:
                count +=1
            gse.append(e_min)
            
            r = (qaoa_energy - e_max) / (e_min - e_max)
            r_n.append(r)
                
            fes = 0
            for s, c in qaoa_result.items():
                fes += check_feasibility(model, s) * c / shots
            qaoa_fes.append(fes)
            print(f"r: {r}, probs: {qaoa_result.get(opt_sol,0)/shots}, t: {t_ws_t + t_qaoa_t}")
        print(f'BF fes: {count/N}')
        data['n_qubits'].append(n_qubits) 
        data['success_counter'].append(count/N)
        data['qaoa_FR'].append(qaoa_fes)
        data['qaoa_e'].append(qaoa_e)
        data['r'].append(r_n)
        data['probs'].append(probs_n)
        data['opt_cost'].append(opt_cost)
        data['t_ws'].append(t_ws)
        data['gates'].append(gates)
        data['t_qaoa'].append(t_qaoa)
        data['t_total'].append(t)
        data['n_fev'].append(n_fev)
        data['gse'].append(gse)
    return data

In [6]:
size_grid = [4, 9]

p = 9
N = 10
alpha = 4

data = {}
print('SV_WS_LR')
data['SV_WS_LR'] = run_MCLP(size_grid, instances_dict, p, N, alpha, slack_vars=True, WS=True, LR=True)
print('UP_WS_LR')
data['UP_WS_LR'] = run_MCLP(size_grid, instances_dict, p, N, alpha, slack_vars=False, WS=True, LR=True)
print('UP_LR')
data['UP_LR'] = run_MCLP(size_grid, instances_dict, p, N, alpha, slack_vars=False, WS=False, LR=True)
print('SV_LR')
data['SV_LR'] = run_MCLP(size_grid, instances_dict, p, N, alpha, slack_vars=True, WS=False, LR=True)
print('SV_WS')
data['SV_WS'] = run_MCLP(size_grid, instances_dict, p, N, alpha, slack_vars=True, WS=True, LR=False)
print('UP_WS')
data['UP_WS'] = run_MCLP(size_grid, instances_dict, p, N, alpha, slack_vars=False, WS=True, LR=False)
print('UP')
data['UP'] = run_MCLP(size_grid, instances_dict, p, N, alpha, slack_vars=False, WS=False, LR=False)
print('SV')
data['SV'] = run_MCLP(size_grid, instances_dict, p, N, alpha, slack_vars=True, WS=False, LR=False)

SV_WS_LR
4 places
0 experiment
12 qubits
r: 0.9998727629339853, probs: 0.998, t: 4.282989501953125
1 experiment
12 qubits
r: 0.999913580148148, probs: 0.997, t: 3.6193289756774902
2 experiment
12 qubits
r: 0.9999690337476923, probs: 0.998, t: 4.235220193862915
3 experiment
12 qubits
r: 0.9999238724999999, probs: 0.997, t: 3.836625337600708
4 experiment
12 qubits
r: 0.999747500098039, probs: 0.993, t: 3.89821457862854
5 experiment
12 qubits
r: 0.9998458679706601, probs: 0.994, t: 3.567347288131714
6 experiment
12 qubits
r: 0.9998573953808355, probs: 0.994, t: 4.223507881164551
7 experiment
12 qubits
r: 0.9998971638141809, probs: 0.996, t: 4.237306833267212
8 experiment
12 qubits
r: 0.9998820147420148, probs: 0.996, t: 3.5190467834472656
9 experiment
12 qubits
r: 0.9999486063569683, probs: 0.998, t: 4.3243560791015625
BF fes: 0.8
9 places
0 experiment
22 qubits
r: 0.9997495145631068, probs: 0.987, t: 63.486300230026245
1 experiment
23 qubits
r: 0.9998406125735293, probs: 0.994, t: 159.41

In [8]:
with open(f"data_r_MCLP.txt", "w") as file:
    file.write(str(data))